In [ ]:
"""
RAGShield — Phase 3 (DeepEval variant)
Run Faithfulness, Contextual Precision, and an Answer-Correctness metric over
your saved RAG results, print a score table, and save report.json.

Install:
    pip install deepeval

DeepEval's LLM-as-judge metrics default to OpenAI. Since your pipeline is
Gemini-based, this script points DeepEval's judge at Gemini too (via
deepeval.models.GeminiModel) — one less API key to manage. Swap judge_model
for deepeval.models.GPTModel if you'd rather judge with GPT-4.
"""

import json
import os

# Must be set before deepeval is imported — it reads these at import time.
# Default retry is weak (2 attempts, 5s cap), nowhere near enough to ride out
# a real per-minute quota. This makes DeepEval wait much longer and retry
# more before giving up on a 429.
os.environ.setdefault("DEEPEVAL_RETRY_MAX_ATTEMPTS", "6")
os.environ.setdefault("DEEPEVAL_RETRY_INITIAL_SECONDS", "10")
os.environ.setdefault("DEEPEVAL_RETRY_EXP_BASE", "2")
os.environ.setdefault("DEEPEVAL_RETRY_CAP_SECONDS", "90")

# DeepEval also enforces an outer time budget per task (per metric per test
# case) that includes waiting on retries. With the RateLimiter below plus the
# retry settings above, one call can legitimately take a few minutes — so the
# outer budget needs to be generous enough to not kill it mid-wait (that's
# what the bare "TimeoutError" was). Leaving per-attempt override unset makes
# deepeval rely on this outer budget alone rather than double-bounding things.
os.environ.setdefault("DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE", "600")
os.environ.setdefault("DEEPEVAL_TASK_GATHER_BUFFER_SECONDS_OVERRIDE", "30")

import asyncio
import threading
import time
from collections import defaultdict, deque
from pathlib import Path
from statistics import mean

from deepeval import evaluate
from deepeval.evaluate import AsyncConfig, CacheConfig
from deepeval.metrics import ContextualPrecisionMetric, FaithfulnessMetric, GEval
from deepeval.models import GeminiModel
from deepeval.test_case import LLMTestCase, LLMTestCaseParams


# ---------- 0. Client-side rate limiter ----------
# The 429 you hit was a per-MINUTE quota (15 requests/min on the free tier for
# gemini-3.1-flash-lite — confirm your own model's limit at
# https://ai.google.dev/gemini-api/docs/rate-limits, since it varies by model).
# DeepEval's AsyncConfig only paces test-case *launches*, not the individual
# judge calls inside each metric — and a single test case can fire several
# calls (faithfulness alone does 2+). So the limit has to be enforced at the
# actual API-call level, which is what this does: a sliding-window limiter
# that every generate()/a_generate() call waits on before firing.

class RateLimiter:
    """Blocks until fewer than max_calls have fired in the last `period` seconds."""

    def __init__(self, max_calls: int, period: float = 60.0):
        self.max_calls = max_calls
        self.period = period
        self.calls = deque()
        self._lock = threading.Lock()
        self._alock = asyncio.Lock()

    def _wait_time(self) -> float:
        now = time.monotonic()
        while self.calls and now - self.calls[0] > self.period:
            self.calls.popleft()
        if len(self.calls) >= self.max_calls:
            return self.period - (now - self.calls[0])
        return 0.0

    def acquire(self):
        with self._lock:
            wait = self._wait_time()
            while wait > 0:
                time.sleep(wait)
                wait = self._wait_time()
            self.calls.append(time.monotonic())

    async def a_acquire(self):
        async with self._alock:
            wait = self._wait_time()
            while wait > 0:
                await asyncio.sleep(wait)
                wait = self._wait_time()
            self.calls.append(time.monotonic())


# Stay a bit under the published 15/min free-tier cap to leave headroom for
# DeepEval's own retries — otherwise a retry can itself trip the same limit.
RATE_LIMIT_PER_MINUTE = 12
_rate_limiter = RateLimiter(max_calls=RATE_LIMIT_PER_MINUTE, period=60.0)


class RateLimitedGeminiModel(GeminiModel):
    """GeminiModel wrapper that enforces RATE_LIMIT_PER_MINUTE before every call."""

    def generate(self, *args, **kwargs):
        _rate_limiter.acquire()
        return super().generate(*args, **kwargs)

    async def a_generate(self, *args, **kwargs):
        await _rate_limiter.a_acquire()
        return await super().a_generate(*args, **kwargs)


# ---------- 1. Config ----------
RESULTS_PATH = Path("utils/reports/response.json")  
REPORT_PATH = Path("utils/reports/report.json")

# Reads GOOGLE_API_KEY from the environment; hardcode a string here instead if you prefer.
judge_model = RateLimitedGeminiModel(
    model="gemini-3.1-flash-lite",  # matches the model your 429 was for — swap if you use another
    api_key=os.environ.get("GOOGLE_API_KEY"),
    temperature=0,
)

# ---------- 2. Load saved results ----------
# Actual shape per record (RAGShield results.json):
# {
#   "question": "...",
#   "expected_answer": "...",
#   "expected_source": "POL-HR-001",
#   "response": {
#     "query": "...",
#     "answer": "...",
#     "context": "...",       # single string, not a list of chunks
#     "citations": "..."
#   }
# }
with open(RESULTS_PATH, "r", encoding="utf-8") as f:
    records = json.load(f)

# ---------- 3. Build DeepEval test cases ----------
# retrieval_context must be a list of strings — "context" here is one string
# (one retrieved/concatenated passage), so it's wrapped in a single-element list.
# If your pipeline actually retrieves multiple chunks and only joins them into
# one string for display, split on your chunk separator instead so each chunk
# is scored individually by ContextualPrecisionMetric.
test_cases = [
    LLMTestCase(
        input=r["question"],
        actual_output=r["response"]["answer"],
        retrieval_context=[r["response"]["context"]],
        expected_output=r.get("expected_answer"),
    )
    for r in records
]

# ---------- 4. Define metrics ----------
faithfulness_metric = FaithfulnessMetric(model=judge_model, threshold=0.5)
context_precision_metric = ContextualPrecisionMetric(model=judge_model, threshold=0.5)

# DeepEval has no built-in "answer correctness" metric (that's Ragas-specific),
# so it's recreated with G-Eval: an LLM judge scoring actual_output against
# expected_output — same idea as Ragas' answer_correctness.
correctness_metric = GEval(
    name="Answer Correctness",
    criteria=(
        "Determine whether the 'actual output' is factually correct and "
        "semantically equivalent to the 'expected output'. Penalize missing "
        "or contradictory facts; do not penalize differences in phrasing."
    ),
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    model=judge_model,
    threshold=0.5,
)

metrics = [faithfulness_metric, context_precision_metric, correctness_metric]

# ---------- 5. Run evaluation ----------
# Gemini's free-tier quota is easy to blow through: each metric makes several
# judge calls per test case (faithfulness extracts claims, then verifies each
# one; GEval does a full reasoning pass), and DeepEval runs up to 20 test
# cases concurrently by default. max_concurrent + throttle_value spread the
# calls out to stay under your per-minute limit; use_cache/write_cache means
# a run that dies partway through a quota wall picks up from cache on retry
# instead of re-scoring everything.
try:
    result = evaluate(
        test_cases=test_cases,
        metrics=metrics,
        async_config=AsyncConfig(
            max_concurrent=1,   # keep serial — avoids two calls racing to acquire the limiter
            throttle_value=1,   # the RateLimiter above now does the real pacing
        ),
        # On Windows without pywin32, DeepEval's cache file locking falls
        # back to msvcrt, which can't do real shared locks — concurrent
        # writes can race and corrupt the cache file (see the try/except
        # below). Either `pip install "portalocker[win32]"` for proper
        # locking, or set use_cache=False here to skip caching altogether.
        cache_config=CacheConfig(use_cache=True, write_cache=True),
    )
except AttributeError as e:
    # Known deepeval bug (confident-ai/deepeval#768): if a previous run died
    # mid-way (e.g. from a quota 429), the cache file can be left corrupted,
    # and the cache loader returns None instead of an empty cache — which
    # crashes on the next lookup with this exact error. Clear the stale cache
    # and retry once, without caching, so this run isn't blocked by it.
    if "get_cached_api_test_case" in str(e) or "test_cases_lookup_map" in str(e):
        print("Stale/corrupted DeepEval cache detected — clearing it and retrying without cache.")
        for stale in (".deepeval-cache.json", ".temp-deepeval-cache.json"):
            Path(stale).unlink(missing_ok=True)
        result = evaluate(
            test_cases=test_cases,
            metrics=metrics,
            async_config=AsyncConfig(max_concurrent=1, throttle_value=1),
            cache_config=CacheConfig(use_cache=False, write_cache=False),
        )
    else:
        raise

# ---------- 6. Collect per-metric averages ----------
scores_by_metric = defaultdict(list)
for test_result in result.test_results:
    for metric_data in test_result.metrics_data:
        scores_by_metric[metric_data.name].append(metric_data.score)

summary = {
    name: {"average_score": round(mean(scores), 4) if scores else None, "n": len(scores)}
    for name, scores in scores_by_metric.items()
}

# ---------- 7. Print a simple score table ----------
print(f"{'Metric':30s} | {'Avg Score':10s} | {'N':5s}")
print("-" * 50)
for name, stats in summary.items():
    avg = stats["average_score"]
    avg_display = f"{avg}" if avg is not None else "N/A"
    print(f"{name:30s} | {avg_display:<10} | {stats['n']:<5}")

# ---------- 8. Save report.json ----------
report = {
    "summary": summary,
    "per_case": [
        {
            "input": tc.input,
            "actual_output": tc.actual_output,
            "expected_source": r.get("expected_source"),
            "citations": r.get("response", {}).get("citations"),
            "metrics": {
                md.name: {"score": md.score, "reason": md.reason, "success": md.success}
                for md in tr.metrics_data
            },
        }
        for tc, tr, r in zip(test_cases, result.test_results, records)
    ],
}

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(f"\nSaved detailed report to {REPORT_PATH.resolve()}")

C:\Users\PRATIK\AppData\Local\Temp\ipykernel_1920\462283418.py:47: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


✨ You're running DeepEval's latest Faithfulness Metric! (using gemini-3.1-flash-lite (Gemini), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using gemini-3.1-flash-lite (Gemini), 
strict=False, async_mode=True)...

✨ You're running DeepEval's latest Answer Correctness [GEval] Metric! (using gemini-3.1-flash-lite (Gemini), 
strict=False, async_mode=True)...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_3 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_4 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_5 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_6 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                               ┃ Average Score      ┃ Pass Rate                               ┃ Total    │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━ │
│  Faithfulness                         │ 1.00               │ 100.00% | passed=7 | failed=0           │ 7        │
│  Contextual Precision                 │ 1.00               │ 100.00% | passed=7 | failed=0           │ 7        │
│  Answer Correctness [GEval]           │ 0.90               │ 100.00% | passed=7 | failed=0           │ 7        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=2272128;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 12.91s | token cost: 0.00139525 USD)
» Test Results (7 total tests):
   » Pass Rate: 100.0% | Passed: 7 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Metric                         | Avg Score  | N    
--------------------------------------------------
Faithfulness                   | 1.0        | 7    
Contextual Precision           | 1.0        | 7    
Answer Correctness [GEval]     | 0.9        | 7    

Saved detailed report to E:\RAGShield\report.json
